In [1]:
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever

raw_documents = [
    Document(page_content="보안 규정: 업무용 PC는 3개월마다 비밀번호를 변경해야 하며 특수문자를 포함해야 합니다."),
    Document(page_content="원격 근무 가이드: 재택 근무 시 반드시 사내 VPN을 연결하여 사내 인트라넷에 접속해야 합니다."),
    Document(page_content="장비 지급 규정: 신규 입사자에게는 노트북 1대와 외장 모니터 1대가 기본 지급됩니다."),
    Document(page_content="계정 관리: 사내 계정 5회 연속 로그인 실패 시 계정이 잠기며 IT 지원팀에 잠금 해제를 요청해야 합니다.")
]

def custom_tokenizer(text):
    return text.split()

bm25_retriever = BM25Retriever.from_documents(
    documents=raw_documents,
    preprocess_func=custom_tokenizer
)

bm25_retriever.k = 2

C:\Users\yonggyo\AppData\Local\Temp\ipykernel_11060\2877887076.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


In [4]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_postgres import PGVector

load_dotenv()

connection_url = os.getenv("DATABASE_URL")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = PGVector.from_documents(
    documents=raw_documents,
    embedding=embeddings,
    connection=connection_url,
    collection_name="it_regulations_collection",
    use_jsonb=True
)

vector_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

In [6]:
from langchain_classic.retrievers import EnsembleRetriever

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5]
)

query = "외부 장소에서 회사 시스템 연결 시 VPN 보안 수칙"

searched_docs = hybrid_retriever.invoke(query)

for i, doc in enumerate(searched_docs, 1):
    print(f"Rank: {i}, doc:{doc.page_content}")

Rank: 1, doc:보안 규정: 업무용 PC는 3개월마다 비밀번호를 변경해야 하며 특수문자를 포함해야 합니다.
Rank: 2, doc:원격 근무 가이드: 재택 근무 시 반드시 사내 VPN을 연결하여 사내 인트라넷에 접속해야 합니다.
Rank: 3, doc:계정 관리: 사내 계정 5회 연속 로그인 실패 시 계정이 잠기며 IT 지원팀에 잠금 해제를 요청해야 합니다.
